In [ ]:
from pathlib import Path
import os

# Set PROJECT_DATA_DIR before launching Jupyter to use data stored elsewhere.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(PROJECT_ROOT / "data"))).expanduser().resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
script_dir = os.getcwd()

In [ ]:


# csv_path = os.path.join(script_dir, str(DATA_DIR / 'test_identity.csv'))

# test_identity = pd.read_csv(csv_path)



In [ ]:
# csv_path = os.path.join(script_dir, str(DATA_DIR / 'test_transaction.csv'))
# test_transaction = pd.read_csv(csv_path)

In [ ]:
# csv_path = os.path.join(script_dir, str(DATA_DIR / 'train_transaction.csv'))

# train_transaction = pd.read_csv(csv_path)

In [ ]:
# csv_path = os.path.join(script_dir, str(DATA_DIR / 'train_identity.csv'))
# train_identity = pd.read_csv(csv_path)

In [ ]:
# test_identity.head(5)


In [ ]:
# test_transaction.head(5)

1. Feature Removal: Features with >95% missing values are removed to prevent sparse representations that may mislead ensemble learners.
2. Strategic Imputation: For categorical features with <95% missing values, we create explicit
"missing" categories to capture the informational content of missingness patterns.
3. Numerical Imputation: Numerical features employ median imputation within fraud/legitimate
groups separately to preserve class-specific distributions.
4. Missingness Indicators: Binary indicators are created for features with >20% missing values to
capture missingness patterns as potential fraud signals.

In [ ]:
# df = test_transaction.copy()

# missing_frac = df.isna().mean().sort_values(ascending=False)

In [ ]:
# threshold = 0.95 

# # Columns to DROP: missing >= 95%
# drop_cols = missing_frac[missing_frac >= threshold].index
# df_reduced = df.drop(columns=drop_cols)

# print(f"Original shape: {df.shape}")
# print(f"Dropped {len(drop_cols)} columns with >= {threshold:.0%} missingness")
# print(f"New shape: {df_reduced.shape}")

# # Optional: keep a quick summary table
# missing_summary = (missing_frac * 100).rename("missing_%").to_frame()
# missing_summary["to_drop_ge_95%"] = missing_summary["missing_%"] >= (threshold * 100)
# missing_summary.head(20)

In [ ]:
# assert "TransactionID" in test_transaction.columns
# assert "TransactionID" in test_identity.columns

# # dedupe duplicates before merge.
# test_identity_dedup = test_identity.drop_duplicates(subset=["TransactionID"])


In [ ]:
# # Combine
# test_merged = test_transaction.merge(
#     test_identity_dedup,
#     on="TransactionID",
#     how="left",
#     validate="one_to_one"  # change to "one_to_many" if identity has legit duplicates
# )

# test_merged.to_csv(os.path.join(script_dir, str(DATA_DIR / 'test_merged.csv')), index=False)

In [ ]:
# # Combine
# train_merged = train_transaction.merge(
#     train_identity,
#     on="TransactionID",
#     how="left",
#     validate="one_to_one"  # change to "one_to_many" if identity has legit duplicates
# )

# train_merged.to_csv(os.path.join(script_dir, str(DATA_DIR / 'train_merged.csv')), index=False)

In [ ]:
# test_merged = pd.read_csv(os.path.join(script_dir, str(DATA_DIR / 'test_merged.csv')))
# train_merged = pd.read_csv(os.path.join(script_dir, str(DATA_DIR / 'train_merged.csv')))

In [ ]:
# print("test_transaction:", test_transaction.shape)
# print("test_identity:", test_identity.shape)
# print("test_merged:", test_merged.shape)


# print("train_transaction:", train_transaction.shape)
# print("train_identity:", train_identity.shape)
# print("train_merged:", train_merged.shape)

In [ ]:
# # Quick sanity checks
# print("Missing identity rows:", test_merged["TransactionID"].isna().sum())  # should be 0
# id_cols = [c for c in test_identity.columns if c != "TransactionID"]
# print("Rows without identity info:", test_merged[id_cols].isna().all(axis=1).sum())

In [ ]:
# id_cols = [c for c in test_identity.columns if c != "TransactionID"]

# has_identity = ~test_merged[id_cols].isna().all(axis=1)
# print("Total rows:", len(test_merged))
# print("Has any identity info:", has_identity.sum())
# print("No identity info:", (~has_identity).sum())
# print("Pct with identity:", has_identity.mean())

In [ ]:
# # Count transactions labeled as fraud
# if 'isFraud' in train_merged.columns:
#     total = len(train_merged)
#     num_fraud = train_merged['isFraud'].sum()
#     num_non_fraud = total - num_fraud

#     print(f"Number of fraud transactions: {num_fraud}")
#     print(f"Number of non-fraud transactions: {num_non_fraud}")
#     print(f"Total transactions: {total}")
#     print(f"Percentage fraud: {num_fraud/total:.2%}")

#     # pie chart of fraud vs non-fraud
#     fraud_counts = train_merged['isFraud'].value_counts()
#     plt.figure(figsize=(6, 6))
#     plt.pie(fraud_counts, labels=['Non-Fraud', 'Fraud'], autopct='%1.1f%%', startangle=90, colors=['skyblue', 'salmon'])
#     plt.title('Distribution of Fraudulent vs Non-Fraudulent Transactions')
#     plt.axis('equal')  # Equal aspect ratio ensures that pie chart is circular.
#     plt.show()
# else:
#     print("Column 'isFraud' not found in train_merged.")

In [ ]:
train_merged.head(5)

In [ ]:
missing_percent = (train_merged.isnull().sum() / len(train_merged)) * 100
missing_percent = missing_percent.sort_values(ascending=False)
print("Missing values percentage by column (top 20):")
missing_percent.head(20)

In [ ]:
if (missing_percent > 95).any():
    drop_cols = missing_percent[missing_percent >= 95].index
    train_merged_dropped_columns = train_merged.drop(columns=drop_cols)


In [ ]:
missing_percent = (train_merged.isnull().sum() / len(train_merged)) * 100
missing_percent = missing_percent.sort_values(ascending=False)
print("Missing values percentage by column (top 20):")
missing_percent.head(20)

In [ ]:
train_merged.dtypes

In [ ]:
#Transaction ID, card1, and addr1 should be treated as categorical variables, even if they are numeric. 
# also check card2, card3, card4, card5, card6 for categorical nature. and addr2


for col in train_merged.columns:
    print(f"{col}: {train_merged[col].nunique()} unique values")



In [ ]:
num_cols = train_merged.select_dtypes(include=["int64","float64"]).columns
cat_cols = train_merged.select_dtypes(include=["object"]).columns

In [ ]:
# for col in num_cols:
#     train_merged[col + "_missing"] = train_merged[col].isna().astype(int)
#     train_merged[col] = train_merged[col].fillna(train_merged[col].median())

In [ ]:
# for col in cat_cols:
#     train_merged[col] = train_merged[col].fillna("unknown")

In [ ]:
# train_merged.to_csv(os.path.join(script_dir, str(DATA_DIR / 'train_merged_cleaned.csv')), index=False)

In [ ]:
df = pd.read_csv(str(DATA_DIR / 'train_merged.csv'))

In [ ]:
# Drop missing columns

missing_ratio = df.isnull().mean()
df = df.loc[:, missing_ratio < 0.95]

In [ ]:
#drop low variance columns (only 1 unique value)

low_var_cols = [col for col in df.columns if df[col].nunique() <= 1]
df.drop(columns=low_var_cols, inplace=True)

In [ ]:
df.drop(columns=["TransactionID"], inplace=True)

In [ ]:
categorical = []
numerical = []

for col in df.columns:
    if col == "isFraud":
        continue
        
    unique_vals = df[col].nunique()
    
    if df[col].dtype == "object":
        categorical.append(col)
        
    elif unique_vals < 50:
        categorical.append(col)
        
    else:
        numerical.append(col)

In [ ]:
known_cat = ["card", "addr", "email", "Product", "M"]

for col in df.columns:
    if any(k in col for k in known_cat):
        categorical.append(col)

categorical = list(set(categorical))
numerical = [col for col in df.columns if col not in categorical + ["isFraud"]]

In [ ]:
for col in numerical:
    df[col + "_missing"] = df[col].isna().astype(int)
    df[col] = df[col].fillna(df[col].median())

In [ ]:
for col in categorical:
    df[col] = df[col].fillna("unknown")

In [ ]:
def encode_train_val(train_df, val_df, categorical_cols):
    encoders = {}
    for col in categorical_cols:
        # Fit on train only; unseen categories in val -> -1
        train_col = train_df[col].astype(str)
        categories = train_col.unique()
        mapping = {k: i for i, k in enumerate(categories)}
        train_df[col] = train_col.map(mapping).astype('int32')
        val_df[col] = val_df[col].astype(str).map(mapping).fillna(-1).astype('int32')
        encoders[col] = mapping
    return train_df, val_df, encoders


Merge → Drop bad features → Define types → Handle missing (+ flags)
→ Encode → Handle imbalance → Train LGBM → Evaluate (AUC)

In [ ]:
df = pd.read_csv(str(DATA_DIR / 'train_merged_submission.csv'))

In [ ]:
# ORIGINAL (leaky) pattern that gave ~0.96 AUC

# Identify categorical columns earlier in the notebook...
from sklearn.preprocessing import LabelEncoder

# Leakage: fit encoders on full dataset before split
for col in categorical:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

scale_pos_weight = (df["isFraud"] == 0).sum() / (df["isFraud"] == 1).sum()

from sklearn.model_selection import train_test_split

X = df.drop("isFraud", axis=1)
y = df["isFraud"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

import lightgbm as lgb

model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    max_depth=-1,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
)

from sklearn.metrics import roc_auc_score
y_pred = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_pred)
print("AUC:", auc)


In [ ]:
# UPDATED SPLIT (time-based) + train-only encoding to reduce leakage
# Run this cell instead of the earlier train_test_split block.

# Time-based split to reduce leakage from temporal ordering
df_sorted = df.sort_values('TransactionDT') if 'TransactionDT' in df.columns else df.copy()
X = df_sorted.drop('isFraud', axis=1)
y = df_sorted['isFraud']
if 'TransactionID' in X.columns:
    X = X.drop(columns=['TransactionID'])

# 80/20 time split
split_idx = int(len(df_sorted) * 0.8)
X_train, X_val = X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy()
y_train, y_val = y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()

def encode_train_val(train_df, val_df, categorical_cols):
    encoders = {}
    for col in categorical_cols:
        # Fit on train only; unseen categories in val -> -1
        train_col = train_df[col].astype(str)
        categories = train_col.unique()
        mapping = {k: i for i, k in enumerate(categories)}
        train_df[col] = train_col.map(mapping).astype('int32')
        val_df[col] = val_df[col].astype(str).map(mapping).fillna(-1).astype('int32')
        encoders[col] = mapping
    return train_df, val_df, encoders

# Encode categoricals using train only
X_train, X_val, cat_encoders = encode_train_val(X_train, X_val, categorical)

# Class weight based on training labels only
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()


In [ ]:
import lightgbm as lgb

model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    max_depth=-1,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
)

In [ ]:
from sklearn.metrics import roc_auc_score

y_pred = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_pred)

print("AUC:", auc)

In [ ]:
# GROUPED SPLIT (entity-based) to reduce entity leakage
# Uses groups built from card1+addr1+P_emaildomain when available
from sklearn.model_selection import GroupShuffleSplit

df_sorted = df.sort_values('TransactionDT') if 'TransactionDT' in df.columns else df.copy()
X = df_sorted.drop('isFraud', axis=1)
y = df_sorted['isFraud']
if 'TransactionID' in X.columns:
    X = X.drop(columns=['TransactionID'])

# Build group id from high-leakage identity fields if present
group_cols = [c for c in ['card1','addr1','P_emaildomain'] if c in df_sorted.columns]
if group_cols:
    groups = df_sorted[group_cols].astype(str).agg('_'.join, axis=1)
else:
    # Fallback: each row is its own group (not ideal but avoids crash)
    groups = df_sorted.index.astype(str)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups=groups))
X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()

# Encode categoricals using train only
X_train, X_val, cat_encoders = encode_train_val(X_train, X_val, categorical)

# Class weight based on training labels only
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()


In [ ]:
import lightgbm as lgb

model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    max_depth=-1,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
)

In [ ]:
from sklearn.metrics import roc_auc_score

y_pred = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_pred)

print("AUC:", auc)

In [ ]:
# STRICTEST SPLIT: time-based holdout + no entity overlap
# 1) Sort by time
# 2) Hold out last 20% by time
# 3) Remove any val rows whose group appears in train
from sklearn.model_selection import GroupShuffleSplit

df_sorted = df.sort_values('TransactionDT') if 'TransactionDT' in df.columns else df.copy()
X_all = df_sorted.drop('isFraud', axis=1)
y_all = df_sorted['isFraud']
if 'TransactionID' in X_all.columns:
    X_all = X_all.drop(columns=['TransactionID'])

# Build group id from stronger identity fields if present
group_cols = [c for c in ['card1','card2','addr1','addr2','P_emaildomain','DeviceInfo'] if c in df_sorted.columns]
if group_cols:
    groups = df_sorted[group_cols].astype(str).agg('_'.join, axis=1)
else:
    groups = df_sorted.index.astype(str)

# Time holdout (last 20%)
split_idx = int(len(df_sorted) * 0.8)
X_train = X_all.iloc[:split_idx].copy()
y_train = y_all.iloc[:split_idx].copy()
groups_train = groups.iloc[:split_idx]

X_val = X_all.iloc[split_idx:].copy()
y_val = y_all.iloc[split_idx:].copy()
groups_val = groups.iloc[split_idx:]

# Remove any validation rows with groups seen in train
train_group_set = set(groups_train.tolist())
mask_val = ~groups_val.isin(train_group_set)
X_val = X_val[mask_val]
y_val = y_val[mask_val]

print(f'Val size after removing overlapping groups: {len(X_val)}')

# Encode categoricals using train only
X_train, X_val, cat_encoders = encode_train_val(X_train, X_val, categorical)

# Class weight based on training labels only
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()


In [ ]:
import lightgbm as lgb

model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    max_depth=-1,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
)

In [ ]:
from sklearn.metrics import roc_auc_score

y_pred = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_pred)

print("AUC:", auc)